# 02 · Which problem is worth solving?

The brief left the problem open, so I asked one question of each candidate: does the outcome change
depending on things we know in advance? If it doesn't, no model can predict it, and a list that
looks clever would just be noise.

In [1]:
import sys

sys.path.insert(0, "../scripts")

import pandas as pd

from quote_followup.config import load_settings
from quote_followup.data import load_quotes

pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 100)
settings = load_settings("../config.toml")
quotes = load_quotes(settings.data_dir)

import explore

## Four candidates

For each one: how often it happens, and its rate in the lowest and highest group of each thing we know in advance.

In [2]:
explore.outcome_scan(settings.data_dir).round(3)

,outcome,overall,split by,lowest group,highest group
0,invoice never paid,0.238,customer_type,0.230,0.259
1,invoice never paid,0.238,acquisition_channel,0.228,0.245
2,invoice never paid,0.238,workflow_type,0.233,0.241
3,invoice never paid,0.238,industry,0.211,0.247
4,invoice never paid,0.238,plan_tier,0.234,0.239
5,invoice never paid,0.238,amount,0.227,0.252
6,invoice never paid,0.238,payment_terms,0.228,0.251
7,visit cancelled,0.097,workflow_type,0.073,0.102
8,visit cancelled,0.097,industry,0.081,0.112
9,visit cancelled,0.097,reschedule_count,0.000,0.229


- **Invoices that never get paid.** My first idea, since getting paid is a real worry for owners.
  But the rate sits between 21% and 26% however you slice it. There's nothing for a model to learn.
- **Cancelled visits.** These only move with rescheduling, which happens after the booking, so it
  can't warn anyone in time.
- **Businesses leaving Jobber.** Only 30 of 500 left, too few to learn from.
- **Quotes being accepted.** This moves a lot: 55% to 79% by type of work, 58% to 76% by how the
  customer found the business, and 61% to 73% by how quickly the quote went out.

So, quotes.

## How quotes end

In [3]:
explore.quote_endings(quotes)

,quotes,median_amount,total_amount,share
status,,,,
accepted,15863,352.13,19564026.81,0.635
declined,3630,390.16,5234922.87,0.145
expired,4063,1202.21,9174971.30,0.163
open,1444,411.65,2089030.78,0.058


Declined and expired together are a third of the quotes with a known ending, worth \$14.4M.

In [4]:
explore.reply_times(quotes)

,count,mean,std,min,25%,50%,75%,90%,max
status,,,,,,,,,
accepted,15863.0,2.70,3.45,0.04,1.04,1.38,3.29,6.21,45.38
declined,3630.0,2.88,3.47,0.04,1.08,2.08,3.38,6.33,45.08


Replies come fast. Half of all yeses arrive within about a day and a half, and nos a little later.
Expired quotes never get a reply at all. So how long a quote has gone without a reply tells us
something about how it will end, and that idea is at the heart of the model.

## How many quotes is an owner waiting on?

Every morning at 8am from September to December, I counted each business's quotes that had gone
out, had no reply yet and hadn't expired.

In [5]:
waiting = explore.waiting_per_morning(quotes, "2024-09-01", "2024-12-20")
print(waiting.describe().round(2).to_string())
print(f"business-mornings with 2 or more quotes waiting: {(waiting >= 2).sum():,}")

count    42020.00
mean         2.24
std          1.52
min          1.00
25%          1.00
50%          2.00
75%          3.00
max         21.00
business-mornings with 2 or more quotes waiting: 25,460


On a morning when a business has anything waiting, it's usually two quotes, and sometimes as many as
21. "Which one do I call first?" comes up on tens of thousands of business-mornings.

## Does each business need its own model?

If some businesses simply win more than others, beyond what their mix of work and customers
explains, one shared model would be wrong for them. I predicted each business's wins from its mix
alone and checked whether the real numbers scatter more than chance would.

In [6]:
explore.business_spread(quotes)

{'scatter compared with chance': 1.1, 'p-value': 0.061}

Businesses scatter about as much as luck alone would produce (a ratio close to 1, and a p-value
above the usual 0.05 bar). So one shared model can serve every business, which is lucky, because a
typical business sends only about 50 quotes a year.